In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ayushuttarwar/gpt-training-data/input.txt


In [2]:
with open("/kaggle/input/datasets/ayushuttarwar/gpt-training-data/input.txt", 'r', encoding="utf-8") as f:
    text = f.read();

In [3]:
print("length of dataset (in chars): ", len(text))

length of dataset (in chars):  1115394


In [4]:
# sample
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
# all unique chars in dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


## Tokenization

- finding the unique chars and sorting them
- then assigning a integer to each unique character

In [6]:
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # string to list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # list of int to string

encoded_text = encode("new learning")
print("encoding: ", encoded_text)
print("decoding: ", decode(encoded_text))

encoding:  [52, 43, 61, 1, 50, 43, 39, 56, 52, 47, 52, 45]
decoding:  new learning


From now on whole implementation will be done using pytorch, because it makes development easier as it provides builtin functions for our usecase

In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
# splitting dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8 # chunk size

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print("when input is ", context, " the target: ", target)

when input is  tensor([18])  the target:  tensor(47)
when input is  tensor([18, 47])  the target:  tensor(56)
when input is  tensor([18, 47, 56])  the target:  tensor(57)
when input is  tensor([18, 47, 56, 57])  the target:  tensor(58)
when input is  tensor([18, 47, 56, 57, 58])  the target:  tensor(1)
when input is  tensor([18, 47, 56, 57, 58,  1])  the target:  tensor(15)
when input is  tensor([18, 47, 56, 57, 58,  1, 15])  the target:  tensor(47)
when input is  tensor([18, 47, 56, 57, 58,  1, 15, 47])  the target:  tensor(58)


In [11]:
torch.manual_seed(68)
batch_size = 4
block_size = 8

# generate batch (data loader)
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb =  get_batch('train')
print('intputs: ')
print(xb.shape)
print(xb)

print("targets: ")
print(yb.shape)
print(yb)

print('-'*10)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target is {target}")

intputs: 
torch.Size([4, 8])
tensor([[39,  1, 41, 59, 54,  6,  0, 32],
        [43, 61, 47, 57,  0, 14, 43, 41],
        [ 6,  1, 47, 52, 44, 53, 50, 42],
        [59, 50,  1, 57, 53,  1, 40, 50]])
targets: 
torch.Size([4, 8])
tensor([[ 1, 41, 59, 54,  6,  0, 32, 53],
        [61, 47, 57,  0, 14, 43, 41, 53],
        [ 1, 47, 52, 44, 53, 50, 42,  1],
        [50,  1, 57, 53,  1, 40, 50, 47]])
----------
when input is [39] the target is 1
when input is [39, 1] the target is 41
when input is [39, 1, 41] the target is 59
when input is [39, 1, 41, 59] the target is 54
when input is [39, 1, 41, 59, 54] the target is 6
when input is [39, 1, 41, 59, 54, 6] the target is 0
when input is [39, 1, 41, 59, 54, 6, 0] the target is 32
when input is [39, 1, 41, 59, 54, 6, 0, 32] the target is 53
when input is [43] the target is 61
when input is [43, 61] the target is 47
when input is [43, 61, 47] the target is 57
when input is [43, 61, 47, 57] the target is 0
when input is [43, 61, 47, 57, 0] the tar

## Building bi-gram language model

In [12]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(68)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # this will be like our weight matrix

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (Batch, Time, Channels-vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # now idx = (B, T) array
        for _ in range(max_new_tokens):
            # make predictions
            logits, loss = self(idx) # calls forward function
            # consider only last time step
            logits = logits[:, -1, :] # = (B, C)
            # softmax for probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to running sequence
            idx = torch.cat((idx, idx_next), dim=1) #(B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
out = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.7091, grad_fn=<NllLossBackward0>)


In [13]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


'pRgE;AUffLDd.Ra;oY-jOLZKga.!ofd!'slQT;f:O?kXicgU , J?
;hvbiSV;yckMB3iN:bad'gq.Ft?wQd&$:.Y'ZEDZQEzxr


In [14]:
# optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [15]:
batch_size = 32
for steps in range(10000):
    # sample a batch
    xb, yb = get_batch('train')

    #evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) # reseting the previous gradients to 0
    loss.backward() # getting the gradients for current pass
    optimizer.step() # updating the weights

print(loss.item())

2.3573310375213623


In [16]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))


INosthigseth
CI t'd fietre peas chor ise asus ad ow fr thandess.
IOFangetll is heresurd ch,ftrpe re thenop.
Ayouly tcanow,

The beld foupe Sorcathan himy I:
Anlawe hounssen I s, myord, thantoreld weanlder withased mer;
PUCAULORINvouthyld owanesodivody I.
Le'd t ouelathe sires, tit I te me, per, the.
PEds,



ELUSSun cese t l?'d thee m, ve tone t Thear, r u'shee toushy pason:
A f ff brillesp stI zer t My:
NGfomenggas tes foru yoolis:
MPRE w meny ariostsoathuem.
fr, wngaige mar, w, tise aran doMAU


> on the postive side we've improved but to be honest we need to improve much more

**This is the simplest possible model, where prediction is based on last token, no interaction or no context**

## self-attention

In [17]:
# # example
# torch.manual_seed(1337)
# B, T, C = 4, 8, 2 #batch, time, channels
# x = torch.randn(B, T, C)
# x.shape

In [18]:
# # we want x[b, t] = mean_(i<=t) x[b, i ]
# xbow = torch.zeros((B, T, C))
# for b in range(B):
#     for t in range(T):
#         xprev = x[b, :t+1] #(t, C)
#         xbow = torch.mean(xprev, 0)
# xbow.shape

In [19]:
# wei = torch.tril(torch.ones(T, T))
# wei = wei / wei.sum(1, keepdim=True)
# xbow2 = wei @ x # (T, T) @ (B, T, C) --> (B, T, T) @ (B, T, C) --> (B, T, C)
# torch.allclose(xbow, xbow2)

In [20]:
# # way 3
# tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
# wei = wei.masked_fill(tril == 0, float('-inf'))
# wei = F.softmax(wei, dim=-1)
# wei

In [21]:
# self attention
torch.manual_seed(1113)
B,T,C = 4, 8, 32
x = torch.randn(B, T, C)

# single head self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16


wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
# out = wei@x

out.shape

torch.Size([4, 8, 16])

In [22]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [23]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3841, 0.6159, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5174, 0.1592, 0.3234, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0619, 0.0586, 0.0018, 0.8776, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0399, 0.1634, 0.5685, 0.0748, 0.1534, 0.0000, 0.0000, 0.0000],
        [0.0821, 0.0826, 0.0685, 0.1458, 0.1209, 0.5001, 0.0000, 0.0000],
        [0.3321, 0.0381, 0.1408, 0.1639, 0.0067, 0.1515, 0.1668, 0.0000],
        [0.0089, 0.2630, 0.5253, 0.0154, 0.0314, 0.0055, 0.1159, 0.0347]],
       grad_fn=<SelectBackward0>)

In [24]:
# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

In [25]:
torch.manual_seed(1729)

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [26]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [27]:
# single head
class Head(nn.Module):
    """ one head self attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        # compute attention scores
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) --> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, C)
        out = wei @ v # (B, T, T) @ ((B, T, C) --> (B, T, C)
        return out

In [28]:
# multi head
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention (parallel) """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ simple MLP linear layer with non-linearity """
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [29]:
# transformer block
class Block(nn.Module):
    """ communication followed by computation """
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed//n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [31]:
# simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        # self.sa_heads = MultiHeadAttention(4, n_embed//4) # i.e., 4 heads of 8D self-attention
        # self.ffwd = FeedForward(n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=4) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed) # final layer norm
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        token_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = token_emb + pos_emb #(B, T, C)
        x = self.blocks(x) # (B, T, C)
        x = self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) #(B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #crop idx to last size(block) tokens
            idx_cond = idx[:, -block_size]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [32]:
model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.2765, val loss 4.2729
step 500: train loss 1.9864, val loss 2.0658
step 1000: train loss 1.5965, val loss 1.7629


KeyboardInterrupt: 

In [ ]:
# open('output.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))